# Global Audience Distribution of Anime Fans

In this notebook, we analyze the global distribution of anime fans based on user profile data. We will extract location information from user profiles, clean the data, and visualize the results on a world map using a logarithmic scale to better represent the distribution.

## Import necessary libraries

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np
import lib.dbconnection as dbc
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)


### 1. Database Connection

In [2]:
engine = dbc.create_db_engine()
print("Connessione al database stabilita.")

Connessione al database stabilita.


 ---

### 2. Data Retrieval

In [3]:
query = "SELECT location FROM profiles WHERE location IS NOT NULL"
df_profiles = pd.read_sql(query, engine)

print(f"Loaded {len(df_profiles)} profiles with location.")
df_profiles.head()

Loaded 337154 profiles with location.


,location
0,South Korea
1,United States
2,Mexico
3,Spain
4,Japan


### 3. Data Cleaning and Country Extraction

In [4]:
def extract_user_country(location):
    if pd.isna(location):
        return None

    parts = str(location).split(',')
    country = parts[-1].strip()


    if country.lower() in ['usa', 'united states of america', 'us']:
        return 'United States'
    if country.lower() in ['uk', 'england', 'scotland']:
        return 'United Kingdom'

    return country

df_profiles['Country'] = df_profiles['location'].apply(extract_user_country)

df_profiles = df_profiles[df_profiles['Country'].str.len() > 2]

### 3. User Count and Logarithmic Scaling

In [5]:
user_counts = df_profiles['Country'].value_counts().reset_index()
user_counts.columns = ['Country', 'User_Count']

user_counts = user_counts[user_counts['User_Count'] > 10]

user_counts['Log_User_Count'] = np.log10(user_counts['User_Count'])

user_counts.head(10)

,Country,User_Count,Log_User_Count
0,Japan,98316,4.992624
1,United States,65240,4.814514
2,Germany,26348,4.420748
3,United Kingdom,19520,4.290480
4,Thailand,16416,4.215267
5,Argentina,13230,4.121560
6,China,13196,4.120442
7,Spain,12916,4.111128
8,France,10041,4.001777
9,Australia,9691,3.986369


### 5. Choropleth Map Creation

In [6]:
fig = px.choropleth(
    user_counts,
    locations="Country",
    locationmode='country names',
    color="Log_User_Count",
    hover_data=["User_Count"],

    color_continuous_scale="Viridis",
    title="Global Distribution of Anime User(Logarithmic Scale)",
    labels={'Log_User_Count': 'Log10(Users)', 'User_Count': 'Utenti Totali'}
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='equirectangular'
    ),
    margin={"r":0,"t":40,"l":0,"b":0}
)
fig.write_html("../../graphs/anime_users_world_log.html")

fig.show()